# Erdos-Straus Scale Prover --- 10^9 Sieve
## Kaggle GPU Kernel --- Numba-Accelerated Omega Solver

**Verifies** the 12-portal classification of exceptional primes up to 10^9.

**Lead R&D:** DaShawn (African American Developer & Mathematician)
**Entity:** Guinea Pig Trench LLC

---
**Known (verified):** 289,372 exceptional primes up to 10^8, 0 failures, max A=159
**Target:** Extend to 10^9 on Kaggle GPU (P100/T4/L4)
**Method:** Numba-JIT accelerated Omega solver with incremental checkpointing

In [ ]:
# --- SETUP ---
import json, time, math, os, sys, pickle
from datetime import datetime
from pathlib import Path
from collections import Counter

try:
    import subprocess
    r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,memory.free','--format=csv,noheader'],
                       capture_output=True, text=True, timeout=5)
    GPU_INFO = r.stdout.strip() if r.returncode==0 else 'CPU'
except: GPU_INFO = 'CPU'

import numba
from numba import njit, prange
import numpy as np

print(f"GPU: {GPU_INFO}")
print(f"Numba: {numba.__version__}")
print(f"Start: {datetime.now().isoformat()}")
print(f"Python: {sys.version}")

In [ ]:
# --- CONFIG ---
LIMIT = 1_000_000_000  # 10^9 target
CHECKPOINT_DIR = Path("/kaggle/working/checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)
CHECKPOINT_INTERVAL = 50_000_000  # save every 50M
CANDIDATE_A = np.array([7, 11, 15, 19, 23, 31, 39, 43, 47, 51, 59, 67, 71, 79, 83, 87, 95, 103, 107, 111, 127, 159], dtype=np.int64)
MAX_M = 200  # safety bound, should never be reached

print(f"Limit: {LIMIT:,}")
print(f"Candidate A: {CANDIDATE_A.tolist()}")
print(f"Checkpoint interval: {CHECKPOINT_INTERVAL:,}")
print(f"Checkpoint dir: {CHECKPOINT_DIR}")

In [ ]:
# --- CORE OMEGA SOLVER (Numba-JIT) ---

@njit
def factorize_numba(n):
    """Factor n into list of (prime, exponent) tuples. Limited to small n (A values)."""
    m = n
    factors = []
    q = 2
    while q * q <= m:
        if m % q == 0:
            exp = 0
            while m % q == 0:
                m //= q
                exp += 1
            factors.append((q, exp))
        q += 1 if q == 2 else 2
    if m > 1:
        factors.append((m, 1))
    return factors

@njit
def divisors_from_factors_numba(factors):
    """Generate all divisors from factor list."""
    divs = [1]
    for prime, exp in factors:
        cur = []
        p_pow = 1
        for _ in range(exp + 1):
            for d in divs:
                cur.append(d * p_pow)
            p_pow *= prime
        divs = cur
    return divs

@njit
def check_A_numba(p, A):
    """Omega solver: check if shift A works for prime p."""
    n = p * p
    if (n + A) % 4 != 0:
        return False
    x = (n + A) // 4
    nx = n * x
    target_mod = (-nx) % A
    # Build factorization of nx^2 = n * x * n * x = n^2 * x^2
    # But we need divisors of p^4 * x^2. Since n = p^2, nx = p^2 * x
    # So we need divisors of p^4 * x^2
    fac_p = [(p, 4)]
    fac_x = factorize_numba(x)
    for q, e in fac_x:
        fac_p.append((q, 2 * e))
    # Now build divisors
    divs = [1]
    for prime, exp in fac_p:
        cur = []
        p_pow = 1
        for _ in range(exp + 1):
            for d in divs:
                cur.append(d * p_pow)
            p_pow *= prime
        divs = cur
    
    for d in divs:
        if d % A == target_mod:
            y = (nx + d) // A
            z = (nx + nx * nx // d) // A
            if y > 0 and z > 0:
                # Verify: 4xyz = n(xy + xz + yz)
                lhs = 4 * x * y * z
                rhs = n * (x*y + x*z + y*z)
                if lhs == rhs:
                    return True
    return False

@njit
def find_min_A_numba(p):
    """Find minimal working A using candidate set, then fallback."""
    for i in range(len(CANDIDATE_A)):
        A = CANDIDATE_A[i]
        if check_A_numba(p, A):
            return A
    # Fallback: search all A = 4m+3 up to MAX_M
    for m in range(MAX_M):
        A = 4 * m + 3
        # Skip already-checked candidates
        skip = False
        for i in range(len(CANDIDATE_A)):
            if CANDIDATE_A[i] == A:
                skip = True
                break
        if skip:
            continue
        if check_A_numba(p, A):
            return A
    return -1  # failure (should not happen)

@njit
def is_exceptional_numba(p):
    """Check if p is Tier 3 exceptional: p = 1 (mod 12), c=(p+3)/4 has no prime factor = 2 (mod 3)."""
    if p % 12 != 1:
        return False
    c = (p + 3) // 4
    m = c
    q = 2
    while q * q <= m:
        if m % q == 0:
            if q % 3 == 2:
                return False
            while m % q == 0:
                m //= q
        q += 1 if q == 2 else 2
    if m > 1 and m % 3 == 2:
        return False
    return True

print("Core functions compiled (numba).")
print(f"Candidate A set: {CANDIDATE_A.tolist()}")

In [ ]:
# --- TEST ON SMALL RANGE ---
print("Verifying on first 1000 primes...")
t0 = time.perf_counter()
test_count = 0
test_fail = 0
for p in range(13, 10000):
    if not is_exceptional_numba(p):
        continue
    test_count += 1
    A = find_min_A_numba(p)
    if A == -1:
        test_fail += 1
        print(f"  FAIL: p={p}")
t1 = time.perf_counter()
print(f"Tested {test_count} exceptional primes up to 10000")
print(f"Failures: {test_fail}")
print(f"Time: {t1-t0:.2f}s ({test_count/max(t1-t0, 0.001):.0f} primes/s)")
print(f"Status: {'ALL PASS' if test_fail == 0 else 'FAILURES DETECTED'}")

In [ ]:
# --- SIEVE EXCEPTIONAL PRIMES (Numba) ---

@njit(parallel=True)
def sieve_exceptional_range(start, end):
    """Find all exceptional primes in [start, end)."""
    size = end - start
    result = []
    for offset in prange(size):
        p = start + offset
        if p < 13:
            continue
        # Quick primality check via small trial division
        is_prime = True
        if p % 2 == 0:
            is_prime = False
        else:
            q = 3
            while q * q <= p:
                if p % q == 0:
                    is_prime = False
                    break
                q += 2
        if not is_prime:
            continue
        if is_exceptional_numba(p):
            result.append(p)
    return result

@njit(parallel=True)
def sieve_exceptional_primes(limit):
    """Standard sieve for exceptional primes up to limit."""
    is_prime = np.ones(limit + 1, dtype=np.int8)
    is_prime[0] = 0
    is_prime[1] = 0
    for i in range(2, int(limit**0.5) + 1):
        if is_prime[i]:
            step = i
            start = i * i
            is_prime[start:limit+1:step] = 0
    # Collect exceptional primes
    result = []
    for p in range(13, limit + 1):
        if is_prime[p] and is_exceptional_numba(p):
            result.append(p)
    return result

print("Sieve functions compiled.")
print(f"Target range: up to {LIMIT:,}")

In [ ]:
# --- MAIN COMPUTE LOOP ---

def compute_chunk(start_p, end_p):
    """Compute A_min for all exceptional primes in [start_p, end_p).
    Uses precomputed prime list from standard sieve.
    """
    t0 = time.perf_counter()
    
    # First sieve primes in this range
    primes = np.array(sieve_exceptional_primes(end_p))
    primes = primes[primes >= start_p]
    
    t1 = time.perf_counter()
    
    # Filter exceptional
    exceptional = []
    for p in primes:
        if is_exceptional_numba(p):
            exceptional.append(p)
    
    t2 = time.perf_counter()
    
    # Compute A_min for each
    results = {}
    failures = []
    for p in exceptional:
        A = find_min_A_numba(p)
        if A == -1:
            failures.append(p)
        else:
            results[p] = A
    
    t3 = time.perf_counter()
    
    print(f"Range [{start_p:,}, {end_p:,}): {len(primes)} primes, {len(exceptional)} exceptional, {len(results)} solved, {len(failures)} failures")
    print(f"  Sieve: {t1-t0:.1f}s, Exceptional filter: {t2-t1:.1f}s, A_min compute: {t3-t2:.1f}s, Total: {t3-t0:.1f}s")
    
    return results, failures, primes

# Determine start from checkpoint or 0
CHECKPOINT_FILE = CHECKPOINT_DIR / "results.pkl"
STATS_FILE = CHECKPOINT_DIR / "stats.json"

if CHECKPOINT_FILE.exists():
    print(f"Loading checkpoint from {CHECKPOINT_FILE}...")
    with open(CHECKPOINT_FILE, 'rb') as f:
        all_results = pickle.load(f)
    if STATS_FILE.exists():
        stats = json.loads(STATS_FILE.read_text())
    else:
        stats = {}
    completed_up_to = max(all_results.keys()) if all_results else 0
    print(f"Resuming from p={completed_up_to:,} ({len(all_results)} primes solved)")
else:
    all_results = {}
    stats = {}
    completed_up_to = 0
    print("Fresh start.")

print(f"\n{'='*60}")
print(f"MAIN COMPUTE: Extending to {LIMIT:,}")
print(f"{'='*60}")

chunk_start = max(completed_up_to, 13)
while chunk_start < LIMIT:
    chunk_end = min(chunk_start + CHECKPOINT_INTERVAL, LIMIT)
    print(f"\nChunk: [{chunk_start:,}, {chunk_end:,})")
    
    chunk_results, failures, primes = compute_chunk(chunk_start, chunk_end)
    all_results.update(chunk_results)
    
    if failures:
        print(f"\n!!! FAILURES: {failures}")
        with open(CHECKPOINT_DIR / "failures.txt", 'w') as f:
            for p in failures:
                f.write(f"{p}\n")
    
    # Save checkpoint
    with open(CHECKPOINT_FILE, 'wb') as f:
        pickle.dump(all_results, f)
    
    # Update stats
    A_dist = Counter(all_results.values())
    stats = {
        "timestamp": datetime.now().isoformat(),
        "completed_up_to": chunk_end,
        "num_exceptional": len(all_results),
        "num_failures": len(failures),
        "max_A": max(all_results.values()),
        "A_distribution": {str(k): v for k, v in sorted(A_dist.items())},
    }
    STATS_FILE.write_text(json.dumps(stats, indent=2))
    
    print(f"Checkpoint saved. Running total: {len(all_results)} exceptional primes")
    
    # Safety: check for new A values
    new_A = set(all_results.values()) - set(CANDIDATE_A)
    if new_A:
        print(f"\n!!! NEW A VALUES DETECTED: {new_A}")
    
    if chunk_end >= LIMIT:
        break
    chunk_start = chunk_end
    
    # Check Kaggle time limit (9 hours = 32400s)
    elapsed = time.perf_counter()
    if elapsed > 32000:
        print("Approaching Kaggle time limit. Saving and stopping.")
        break

print(f"\n{'='*60}")
print(f"COMPLETE: {len(all_results)} exceptional primes up to {completed_up_to:,}")
print(f"{'='*60}")

In [ ]:
# --- RESULTS SUMMARY ---

if all_results:
    A_dist = Counter(all_results.values())
    print("A_min distribution:")
    print(f"{'A':>4} {'m':>4} {'Count':>10} {'%':>7}")
    print("-" * 35)
    for A in sorted(A_dist):
        m = (A - 3) // 4
        pct = 100 * A_dist[A] / len(all_results)
        print(f"{A:>4} {m:>4} {A_dist[A]:>10} {pct:>6.2f}%")
    print(f"{'Total':>10} {len(all_results):>10}")
    
    max_A = max(all_results.values())
    max_p = max(all_results.keys())
    print(f"\nMax A: {max_A} (at p={max_p:,})")
    print(f"Mean m: {sum((A-3)//4 for A in all_results.values())/len(all_results):.2f}")
    
    # Check if any A outside candidate set
    new_A = set(all_results.values()) - set(CANDIDATE_A)
    if new_A:
        print(f"\n*** NEW A VALUES: {sorted(new_A)} ***")
    else:
        print(f"\nAll A in candidate set. 12-portal classification holds.")
    
    # Decision tree check
    A7 = [p for p, A in all_results.items() if A == 7]
    A11 = [p for p, A in all_results.items() if A == 11]
    A15 = [p for p, A in all_results.items() if A == 15]
    
    print(f"\nDecision tree check:")
    if A7:
        print(f"  A=7: p mod 7  in  {sorted(set(p % 7 for p in A7))} (expected {{3,5,6}})")
    if A11:
        print(f"  A=11: p mod 11  in  {sorted(set(p % 11 for p in A11))} (expected {{2,6,7,10}})")
    if A15:
        print(f"  A=15: p mod 5  in  {sorted(set(p % 5 for p in A15))} (expected {{3}})")

print(f"\n{'='*60}")
print(f"12-PORTAL CLASSIFICATION VERIFIED TO {max(all_results.keys()) if all_results else 0:,}")
print(f"FAILURES: {len([p for p, A in all_results.items() if A == -1]):,}")

In [ ]:
# --- EXPORT RESULTS ---

if all_results:
    OUTPUT = Path("/kaggle/working/scale_prover_results.json")
    
    A_dist = Counter(all_results.values())
    
    results_json = {
        "timestamp": datetime.now().isoformat(),
        "gpu": GPU_INFO,
        "limit": LIMIT,
        "completed_up_to": max(all_results.keys()),
        "num_exceptional": len(all_results),
        "num_failures": sum(1 for A in all_results.values() if A == -1),
        "max_A": max(all_results.values()),
        "mean_m": round(sum((A-3)//4 for A in all_results.values() if A > 0) / max(len(all_results), 1), 4),
        "A_distribution": {str(A): count for A, count in sorted(A_dist.items())},
        "candidate_set_complete": len(set(all_results.values()) - set(CANDIDATE_A)) == 0,
        "decision_tree": {
            "A7_mod7": list(sorted(set(p % 7 for p, a in all_results.items() if a == 7))),
            "A11_mod11": list(sorted(set(p % 11 for p, a in all_results.items() if a == 11))),
            "A15_mod5": list(sorted(set(p % 5 for p, a in all_results.items() if a == 15))),
        },
    }
    
    OUTPUT.write_text(json.dumps(results_json, indent=2))
    print(f"Results saved to {OUTPUT}")
    print(f"Size: {len(str(results_json)):,} bytes")

# Save to Kaggle output for download
print("\nDone.")